# Amazon ML Challenge 2026: Stage 1 Dual-Channel Embeddings (GPU - 2x T4)
- Model: `intfloat/multilingual-e5-small` (118M params, MIT license, 384 dims)
- Dual independent embedding spaces: `name_core` and `addr_core`
- PyTorch FP16 autocast + DataLoader (batch size 512)
- Resumable shard saving (500k chunks) to fit within Kaggle session timeout


In [ ]:
# Environment & GPU Verification
import os, sys, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0), "| Count:", torch.cuda.device_count())

NORM_DIR = Path('/kaggle/input/normalized') if os.path.exists('/kaggle/input/normalized') else Path('/kaggle/working/normalized')
if not NORM_DIR.exists():
    NORM_DIR = Path('work/normalized')

EMB_DIR = Path('/kaggle/working/embeddings') if os.path.exists('/kaggle') else Path('work/embeddings')
EMB_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Dual-Channel Encoder Class
MODEL_NAME = 'intfloat/multilingual-e5-small'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
if DEVICE == 'cuda' and torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

@torch.no_grad()
def encode_texts(texts, prefix="passage: ", batch_size=512, max_len=64):
    formatted = [f"{prefix}{t}" for t in texts]
    all_embs = []
    use_amp = (DEVICE == 'cuda')
    for i in range(0, len(formatted), batch_size):
        batch = formatted[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE, enabled=use_amp):
            out = model(**enc)
            emb = mean_pooling(out, enc['attention_mask'])
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_embs.append(emb.cpu().to(torch.float16).numpy())
    return np.vstack(all_embs)


In [ ]:
# Encode S1 and S2+S3 Target Records
def process_split(split='train'):
    print(f"\n=== Encoding {split.upper()} ===")
    s1_path = NORM_DIR / f"{split}_s1_norm.parquet"
    s2_path = NORM_DIR / f"{split}_s2_norm.parquet"
    s3_path = NORM_DIR / f"{split}_s3_norm.parquet"
    
    if not s1_path.exists():
        print(f"File {s1_path} not found. Skipping.")
        return
        
    df_s1 = pd.read_parquet(s1_path)
    print(f"S1 {split}: {len(df_s1):,} records")
    s1_name_emb = encode_texts(df_s1['name_core'].fillna('').tolist(), prefix="query: ")
    s1_addr_emb = encode_texts(df_s1['addr_core'].fillna('').tolist(), prefix="query: ")
    np.save(EMB_DIR / f"{split}_s1_name_emb.npy", s1_name_emb)
    np.save(EMB_DIR / f"{split}_s1_addr_emb.npy", s1_addr_emb)
    del s1_name_emb, s1_addr_emb, df_s1
    gc.collect()
    
    # Target S2 + S3
    df_s2 = pd.read_parquet(s2_path)
    df_s3 = pd.read_parquet(s3_path)
    df_tgt = pd.concat([df_s2, df_s3], ignore_index=True)
    print(f"Target (S2+S3) {split}: {len(df_tgt):,} records")
    
    # Save target ID index mapping
    df_tgt[['entity_id']].to_parquet(EMB_DIR / f"{split}_target_ids.parquet", index=False)
    
    tgt_name_emb = encode_texts(df_tgt['name_core'].fillna('').tolist(), prefix="passage: ")
    np.save(EMB_DIR / f"{split}_tgt_name_emb.npy", tgt_name_emb)
    del tgt_name_emb
    gc.collect()
    
    tgt_addr_emb = encode_texts(df_tgt['addr_core'].fillna('').tolist(), prefix="passage: ")
    np.save(EMB_DIR / f"{split}_tgt_addr_emb.npy", tgt_addr_emb)
    del tgt_addr_emb, df_s2, df_s3, df_tgt
    gc.collect()

process_split('train')
process_split('test')
print("Stage 1 Embeddings Complete!")
